# Data Profiling - NYC Citi Bike 2019

In [1]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# 1. Dynamically find the project root
current_dir = Path.cwd()
if (current_dir / "data").exists() and (current_dir / "src").exists():
    project_root = current_dir
else:
    project_root = current_dir.parent

# 2. Load the Bronze Parquet file (much faster than scanning 26 CSVs!)
bronze_path = project_root / "data" / "processed" / "trips_raw_bronze.parquet"

if not bronze_path.is_file():
    raise FileNotFoundError(f"Bronze Parquet file not found: {bronze_path}\nRun the ETL pipeline first.")

df = pl.read_parquet(bronze_path)
print(f"✅ Successfully loaded {len(df):,} raw rows.")

✅ Successfully loaded 20,551,697 raw rows.


In [2]:
print("="*50)
print("1. SCHEMA & DATA TYPES")
print("="*50)
for col, dtype in df.schema.items():
    print(f"{col:<25} : {dtype}")

print("\n" + "="*50)
print("2. DATASET SIZE")
print("="*50)
print(f"Total Rows: {len(df):,}")
print(f"Total Columns: {len(df.schema)}")

print("\n" + "="*50)
print("3. SAMPLE DATA (First 5 Rows)")
print("="*50)
display(df.head(5))

1. SCHEMA & DATA TYPES
trip_duration_seconds     : Int64
start_time                : String
stop_time                 : String
start_station_id          : Int64
start_station_name        : String
start_station_lat         : Float64
start_station_lon         : Float64
end_station_id            : Int64
end_station_name          : String
end_station_lat           : Float64
end_station_lon           : Float64
bike_id                   : Int64
user_type                 : String
birth_year                : Int64
gender_id                 : Int64
source_file               : String

2. DATASET SIZE
Total Rows: 20,551,697
Total Columns: 16

3. SAMPLE DATA (First 5 Rows)


trip_duration_seconds,start_time,stop_time,start_station_id,start_station_name,start_station_lat,start_station_lon,end_station_id,end_station_name,end_station_lat,end_station_lon,bike_id,user_type,birth_year,gender_id,source_file
i64,str,str,i64,str,f64,f64,i64,str,f64,f64,i64,str,i64,i64,str
527,"""2019-10-01 00:00:05.6180""","""2019-10-01 00:08:52.9430""",3746,"""6 Ave & Broome St""",40.724308,-74.00473,223,"""W 13 St & 7 Ave""",40.737815,-73.999947,41750,"""Subscriber""",1993,1,"""D:/henkel-citibike-task/data/r…"
174,"""2019-10-01 00:00:15.8750""","""2019-10-01 00:03:10.1680""",3301,"""Columbus Ave & W 95 St""",40.791956,-73.968087,3283,"""W 89 St & Columbus Ave""",40.788221,-73.970416,18264,"""Subscriber""",1992,1,"""D:/henkel-citibike-task/data/r…"
759,"""2019-10-01 00:00:19.8240""","""2019-10-01 00:12:59.7070""",161,"""LaGuardia Pl & W 3 St""",40.72917,-73.998102,174,"""E 25 St & 1 Ave""",40.738177,-73.977387,25525,"""Subscriber""",1995,1,"""D:/henkel-citibike-task/data/r…"
615,"""2019-10-01 00:00:21.0680""","""2019-10-01 00:10:36.6790""",254,"""W 11 St & 6 Ave""",40.735324,-73.998004,477,"""W 41 St & 8 Ave""",40.756405,-73.990026,30186,"""Subscriber""",1992,1,"""D:/henkel-citibike-task/data/r…"
761,"""2019-10-01 00:00:26.3800""","""2019-10-01 00:13:08.3130""",161,"""LaGuardia Pl & W 3 St""",40.72917,-73.998102,174,"""E 25 St & 1 Ave""",40.738177,-73.977387,25597,"""Subscriber""",1992,1,"""D:/henkel-citibike-task/data/r…"


In [3]:
print("="*50)
print("4. MISSING DATA (NULL COUNTS)")
print("="*50)

null_counts = df.select([
    pl.col("*").null_count().name.prefix("null_")
]).transpose(include_header=True, header_name="column", column_names=["null_count"])

null_counts = null_counts.with_columns(
    (pl.col("null_count") / len(df) * 100).round(3).alias("pct_null")
)

# Filter to only show columns with > 0 nulls for a cleaner output
null_counts_filtered = null_counts.filter(pl.col("null_count") > 0)
if len(null_counts_filtered) == 0:
    print("✅ No null values found in any column!")
else:
    print(null_counts_filtered)

4. MISSING DATA (NULL COUNTS)
shape: (4, 3)
┌─────────────────────────┬────────────┬──────────┐
│ column                  ┆ null_count ┆ pct_null │
│ ---                     ┆ ---        ┆ ---      │
│ str                     ┆ u32        ┆ f64      │
╞═════════════════════════╪════════════╪══════════╡
│ null_start_station_id   ┆ 180        ┆ 0.001    │
│ null_start_station_name ┆ 180        ┆ 0.001    │
│ null_end_station_id     ┆ 180        ┆ 0.001    │
│ null_end_station_name   ┆ 180        ┆ 0.001    │
└─────────────────────────┴────────────┴──────────┘


In [4]:
print("="*50)
print("5. DATA QUALITY ISSUES")
print("="*50)

# A. Trip Duration
print("\n--- Trip Duration ---")
duration_stats = df.select([
    pl.col("trip_duration_seconds").min().alias("min_duration_sec"),
    pl.col("trip_duration_seconds").max().alias("max_duration_sec"),
    (pl.col("trip_duration_seconds") <= 0).sum().alias("zero_or_negative_trips"),
    (pl.col("trip_duration_seconds") > 86400).sum().alias("trips_over_24h")
])
print(duration_stats)

# B. Demographics (Birth Year)
print("\n--- Birth Year ---")
birth_stats = df.select([
    pl.col("birth_year").min().alias("min_birth_year"),
    pl.col("birth_year").max().alias("max_birth_year"),
    (pl.col("birth_year") < 1920).sum().alias("birth_year_before_1920"),
    (pl.col("birth_year") > 2010).sum().alias("birth_year_after_2010")
])
print(birth_stats)

# C. Gender Distribution
print("\n--- Gender Distribution ---")
# 0=Unknown, 1=Male, 2=Female
gender_dist = df.group_by("gender_id").agg(pl.len().alias("count")).sort("gender_id")
print(gender_dist)

# D. Station Coordinates (NYC Bounds Check)
print("\n--- Station Coordinates (Start Lat/Lon) ---")
coord_stats = df.select([
    (pl.col("start_station_lat") < 40.4).sum().alias("lat_too_low"),
    (pl.col("start_station_lat") > 41.0).sum().alias("lat_too_high"),
    (pl.col("start_station_lon") < -74.3).sum().alias("lon_too_low"),
    (pl.col("start_station_lon") > -73.7).sum().alias("lon_too_high")
])
print(coord_stats)

# E. Station Coordinate Consistency Check (NEW)
print("\n--- Station Coordinate Consistency ---")
station_coord_check = df.group_by("start_station_id").agg([
    pl.col("start_station_name").n_unique().alias("unique_names"),
    pl.col("start_station_lat").n_unique().alias("unique_lats"),
    pl.col("start_station_lon").n_unique().alias("unique_lons")
]).filter(
    (pl.col("unique_lats") > 1) | (pl.col("unique_lons") > 1)
).sort("unique_lats", descending=True)

conflicting_stations = station_coord_check
print(f"⚠️ Found {len(conflicting_stations)} station IDs with conflicting coordinates.")
if len(conflicting_stations) > 0:
    print("Top 10 examples:")
    print(conflicting_stations.head(10))

5. DATA QUALITY ISSUES

--- Trip Duration ---
shape: (1, 4)
┌──────────────────┬──────────────────┬────────────────────────┬────────────────┐
│ min_duration_sec ┆ max_duration_sec ┆ zero_or_negative_trips ┆ trips_over_24h │
│ ---              ┆ ---              ┆ ---                    ┆ ---            │
│ i64              ┆ i64              ┆ u32                    ┆ u32            │
╞══════════════════╪══════════════════╪════════════════════════╪════════════════╡
│ 61               ┆ 3812666          ┆ 0                      ┆ 6118           │
└──────────────────┴──────────────────┴────────────────────────┴────────────────┘

--- Birth Year ---
shape: (1, 4)
┌────────────────┬────────────────┬────────────────────────┬───────────────────────┐
│ min_birth_year ┆ max_birth_year ┆ birth_year_before_1920 ┆ birth_year_after_2010 │
│ ---            ┆ ---            ┆ ---                    ┆ ---                   │
│ i64            ┆ i64            ┆ u32                    ┆ u32             

In [5]:
print("="*50)
print("6. RELATIONSHIPS (User Type vs Demographics)")
print("="*50)

reliability_check = df.group_by("user_type").agg([
    pl.len().alias("total_trips"),
    pl.col("birth_year").is_null().sum().alias("null_birth_year"),
    (pl.col("gender_id") == 0).sum().alias("unknown_gender"),
    ((2019 - pl.col("birth_year")) < 10).sum().alias("invalid_age") 
]).with_columns([
    (pl.col("null_birth_year") / pl.col("total_trips") * 100).round(2).alias("pct_null_birth_year"),
    (pl.col("unknown_gender") / pl.col("total_trips") * 100).round(2).alias("pct_unknown_gender"),
    (pl.col("invalid_age") / pl.col("total_trips") * 100).round(2).alias("pct_invalid_age")
]).sort("user_type")

print(reliability_check)

print("\n--- Avg Trip Duration by User Type ---")
avg_duration = df.group_by("user_type").agg(
    pl.col("trip_duration_seconds").mean().round(1).alias("avg_duration_sec")
)
print(avg_duration)

6. RELATIONSHIPS (User Type vs Demographics)


shape: (2, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ user_type  ┆ total_trip ┆ null_birth ┆ unknown_g ┆ invalid_a ┆ pct_null_ ┆ pct_unkno ┆ pct_inval │
│ ---        ┆ s          ┆ _year      ┆ ender     ┆ ge        ┆ birth_yea ┆ wn_gender ┆ id_age    │
│ str        ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ r         ┆ ---       ┆ ---       │
│            ┆ u32        ┆ u32        ┆ u32       ┆ u32       ┆ ---       ┆ f64       ┆ f64       │
│            ┆            ┆            ┆           ┆           ┆ f64       ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Customer   ┆ 2872176    ┆ 0          ┆ 1240743   ┆ 0         ┆ 0.0       ┆ 43.2      ┆ 0.0       │
│ Subscriber ┆ 17679521   ┆ 0          ┆ 319565    ┆ 0         ┆ 0.0       ┆ 1.81      ┆ 0.0       │
└────────────┴────────────┴────────────┴───────────┴───────────┴───────────┴─

# Data Quality Findings

Based on the profiling of the consolidated Bronze dataset (`trips_raw_bronze.parquet`), the following data quality characteristics and issues were identified:

### 1. Dataset Overview
*   **Total Records:** 20,551,697 trips.
*   **Missing Data:** Exceptionally clean overall.
    *   `start_station_id`, `start_station_name`, `end_station_id`, and `end_station_name` have exactly 180 nulls each (0.001%).
    *   `birth_year` and `gender_id` have **0 nulls**.

### 2. Trip Duration
*   **Range:** 61 seconds to 3,812,666 seconds (~44 days).
*   **Anomalies:** 6,118 trips (0.03%) exceed 24 hours (86,400 seconds). There are 0 trips with a duration of $\le$ 0 seconds.
*   **Action:** Trips $> 24$ hours will be filtered out, as they likely represent lost bikes, forgotten returns, or system errors rather than actual rides.

### 3. Demographics (Birth Year & Gender)
*   **Birth Year:** Ranges from 1857 to 2003. There are 9,969 records (0.05%) with a birth year before 1920.
    *   **Action:** Cap the valid age range to 12–90 years old (birth years 1929–2007 for 2019 data) to remove data entry errors.
*   **Gender Distribution:** No nulls, but "Unknown" is explicitly encoded as `0`.
    *   **0 (Unknown):** 1,560,308 (7.6%)
    *   **1 (Male):** 14,052,393 (68.4%)
    *   **2 (Female):** 4,938,996 (24.0%)

### 4. Station Coordinates
*   **Bounds Check:** 0 records fall outside the expected NYC bounding box (Lat: 40.4–41.0, Lon: -74.3 to -73.7).
*   **Coordinate Consistency (Drift/Duplicates):** 
    *   **Finding:** Profiling revealed that **60** unique station IDs map to multiple different latitude/longitude combinations (e.g., Station 3305 has 2 different lat/lon pairs). 
    *   **Impact:** If left unhandled, this will create duplicate Primary Keys in the `dim_station` table, breaking the One-to-Many relationships in Power BI and causing incorrect spatial aggregations.
    *   **Action:** During the Transformation phase, we will group by `station_id` and retain only the coordinate pair that appears most frequently for that station, ensuring a clean, unique dimension table.

### 5. User Type Relationships & Reliability
*   **Trip Duration by User Type:**
    *   **Customers:** Average 2,103.3 seconds (~35 minutes).
    *   **Subscribers:** Average 795.5 seconds (~13 minutes).
*   **Data Reliability:** 
    *   **Customers** have a drastically higher rate of `Unknown` gender (43.2%) compared to **Subscribers** (1.81%). 
    *   **Insight:** This perfectly aligns with expected behavior. Subscribers are likely commuters taking short, direct trips and must provide demographic details to create an account. Customers are likely tourists or leisure riders taking longer trips via quick kiosk transactions without providing full profile data.